# 👑 Drop Queen
## Week 1 — Data Loading & Exploratory Data Analysis (EDA)
**CISC 610 — DevOps & MLOps | Mercy University | Spring 2026**

---

### 🎯 Notebook Goals
By the end of this notebook you will have:
- ✅ Connected Google Drive and set up project folders
- ✅ Installed all required libraries
- ✅ Downloaded Amazon Beauty dataset from Kaggle
- ✅ Loaded and explored the data
- ✅ Cleaned and preprocessed the data
- ✅ Created visualizations showing key insights
- ✅ Saved cleaned data to Google Drive

---

> *'GoVirallQ predicts what goes viral. Drop Queen predicts what sells because of it.'*

## 📦 Step 1 — Mount Google Drive
Run this first to connect your Google Drive so your work is saved automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive connected!')

## 📁 Step 2 — Create Project Folder Structure
This creates all the folders Drop Queen needs inside your Google Drive.

In [ ]:
import os

# Base project folder in Google Drive
BASE = '/content/drive/MyDrive/DropQueen'

# Create all folders
folders = [
    f'{BASE}/data/raw',
    f'{BASE}/data/processed',
    f'{BASE}/notebooks',
    f'{BASE}/models',
    f'{BASE}/outputs',
    f'{BASE}/visualizations'
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'✅ Created: {folder}')

print('\n👑 Drop Queen folder structure ready!')

## ⚙️ Step 3 — Install Required Libraries
Install all the libraries needed for Drop Queen.

In [ ]:
# Install required libraries
!pip install prophet mlflow xgboost kaggle pytrends plotly -q

print('✅ All libraries installed!')

## 📥 Step 4 — Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Style settings
plt.style.use('dark_background')
PINK = '#ff3e8a'
GOLD = '#f5c842'
PURPLE = '#9b5de5'
TEAL = '#00c9b1'
COLORS = [PINK, GOLD, PURPLE, TEAL, '#ff85b3', '#fff']

sns.set_palette(COLORS)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print('✅ Libraries imported!')
print(f'📦 Pandas version: {pd.__version__}')
print(f'📦 NumPy version: {np.__version__}')

## 🔑 Step 5 — Connect to Kaggle
**Before running this cell:**
1. Go to [kaggle.com](https://www.kaggle.com) → Your Profile → Settings → API → Create New Token
2. This downloads a `kaggle.json` file to your computer
3. Run the cell below and upload that file when prompted

In [ ]:
from google.colab import files

# Upload your kaggle.json file
print('📂 Please upload your kaggle.json file...')
uploaded = files.upload()

# Set up Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print('✅ Kaggle connected!')

## 📊 Step 6 — Download Amazon Beauty Dataset
Download the Amazon Beauty Reviews dataset from Kaggle.

In [ ]:
import os

RAW_DATA = f'{BASE}/data/raw'

# Download Amazon Beauty Reviews dataset
print('⬇️ Downloading Amazon Beauty dataset...')
!kaggle datasets download -d skillsmuggler/amazon-ratings -p {RAW_DATA} --unzip

# List downloaded files
print('\n📁 Downloaded files:')
for f in os.listdir(RAW_DATA):
    size = os.path.getsize(f'{RAW_DATA}/{f}') / (1024*1024)
    print(f'  📄 {f} ({size:.1f} MB)')

print('\n✅ Dataset downloaded!')

## 🔍 Step 7 — Load & Preview The Data

In [ ]:
# Load the dataset
# Note: Update filename if different after download
DATA_FILE = f'{RAW_DATA}/ratings_Beauty.csv'

print('📂 Loading dataset...')
df = pd.read_csv(DATA_FILE, 
                  names=['user_id', 'product_id', 'rating', 'timestamp'],
                  header=None)

print(f'✅ Dataset loaded!')
print(f'📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\n👀 First 5 rows:')
df.head()

In [ ]:
# Basic info about the dataset
print('📋 Dataset Info:')
print('=' * 50)
df.info()
print('\n📊 Basic Statistics:')
print('=' * 50)
df.describe()

## 🧹 Step 8 — Data Cleaning

In [ ]:
print('🧹 Cleaning data...')
print(f'Original shape: {df.shape}')

# 1. Check for missing values
print(f'\n❓ Missing values:')
print(df.isnull().sum())

# 2. Drop duplicates
before = len(df)
df = df.drop_duplicates()
print(f'\n🗑️ Removed {before - len(df):,} duplicate rows')

# 3. Drop rows with missing values
df = df.dropna()

# 4. Convert timestamp to datetime
df['date'] = pd.to_datetime(df['timestamp'], unit='s')
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['year_month'] = df['date'].dt.to_period('M')

# 5. Filter ratings to valid range (1-5)
df = df[df['rating'].between(1, 5)]

# 6. Filter to recent years (2010+)
df = df[df['year'] >= 2010]

print(f'\n✅ Clean shape: {df.shape}')
print(f'\n📅 Date range: {df["date"].min().date()} to {df["date"].max().date()}')
df.head()

## 📈 Step 9 — Exploratory Data Analysis (EDA)
### 9.1 — Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0a0a')

# Rating distribution
rating_counts = df['rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, 
            color=[PINK, GOLD, PURPLE, TEAL, '#ff85b3'])
axes[0].set_title('👑 Rating Distribution', color='white', fontsize=14, pad=15)
axes[0].set_xlabel('Rating (1-5 Stars)', color='white')
axes[0].set_ylabel('Number of Reviews', color='white')
axes[0].tick_params(colors='white')
axes[0].set_facecolor('#1e1e2e')
for spine in axes[0].spines.values():
    spine.set_edgecolor('#2e2e42')

# Add value labels on bars
for i, v in enumerate(rating_counts.values):
    axes[0].text(i+1, v + 100, f'{v:,}', ha='center', color='white', fontsize=9)

# Rating percentage pie
axes[1].pie(rating_counts.values, 
            labels=[f'{i}⭐' for i in rating_counts.index],
            colors=[PINK, GOLD, PURPLE, TEAL, '#ff85b3'],
            autopct='%1.1f%%',
            textprops={'color': 'white'})
axes[1].set_title('Rating Breakdown %', color='white', fontsize=14, pad=15)
axes[1].set_facecolor('#1e1e2e')

plt.tight_layout()
plt.savefig(f'{BASE}/visualizations/rating_distribution.png', 
            dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print(f'✅ Saved to Google Drive!')

### 9.2 — Review Volume Over Time (Demand Trends)

In [ ]:
# Monthly review volume — proxy for demand
monthly_reviews = df.groupby('year_month').size().reset_index()
monthly_reviews.columns = ['year_month', 'review_count']
monthly_reviews['year_month_str'] = monthly_reviews['year_month'].astype(str)

fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#1e1e2e')

ax.fill_between(range(len(monthly_reviews)), 
                monthly_reviews['review_count'],
                alpha=0.3, color=PINK)
ax.plot(range(len(monthly_reviews)), 
        monthly_reviews['review_count'],
        color=PINK, linewidth=2)

# Mark peak demand months
peak_idx = monthly_reviews['review_count'].idxmax()
ax.scatter(peak_idx, monthly_reviews.loc[peak_idx, 'review_count'],
           color=GOLD, s=100, zorder=5, label='Peak Demand')
ax.annotate(f'Peak: {monthly_reviews.loc[peak_idx, "year_month_str"]}',
            xy=(peak_idx, monthly_reviews.loc[peak_idx, 'review_count']),
            xytext=(peak_idx-10, monthly_reviews.loc[peak_idx, 'review_count']*1.05),
            color=GOLD, fontsize=10)

# X axis labels - show every 12 months
tick_positions = range(0, len(monthly_reviews), 12)
ax.set_xticks(tick_positions)
ax.set_xticklabels([monthly_reviews.loc[i, 'year_month_str'] 
                    for i in tick_positions if i < len(monthly_reviews)], 
                   rotation=45, color='white')

ax.set_title('👑 Drop Queen — Monthly Review Volume (Demand Proxy)', 
             color='white', fontsize=14, pad=15)
ax.set_xlabel('Month', color='white')
ax.set_ylabel('Number of Reviews', color='white')
ax.tick_params(colors='white')
ax.legend(facecolor='#1e1e2e', labelcolor='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2e2e42')

plt.tight_layout()
plt.savefig(f'{BASE}/visualizations/demand_over_time.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print('✅ Demand trend chart saved!')

### 9.3 — Top Products by Review Volume

In [ ]:
# Top 15 most reviewed products
top_products = df['product_id'].value_counts().head(15).reset_index()
top_products.columns = ['product_id', 'review_count']
top_products['short_id'] = top_products['product_id'].str[:10] + '...'

fig, ax = plt.subplots(figsize=(14, 8))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#1e1e2e')

bars = ax.barh(range(len(top_products)), 
               top_products['review_count'],
               color=[PINK if i == 0 else PURPLE for i in range(len(top_products))])

ax.set_yticks(range(len(top_products)))
ax.set_yticklabels(top_products['short_id'], color='white', fontsize=9)
ax.set_title('👑 Top 15 Most Reviewed Beauty Products', 
             color='white', fontsize=14, pad=15)
ax.set_xlabel('Number of Reviews (Demand Indicator)', color='white')
ax.tick_params(colors='white')
ax.invert_yaxis()

# Add value labels
for i, bar in enumerate(bars):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{top_products.loc[i, "review_count"]:,}',
            va='center', color='white', fontsize=9)

for spine in ax.spines.values():
    spine.set_edgecolor('#2e2e42')

plt.tight_layout()
plt.savefig(f'{BASE}/visualizations/top_products.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print('✅ Top products chart saved!')

### 9.4 — Average Rating Per Product (Quality Signal)

In [ ]:
# Products with 100+ reviews — more reliable ratings
product_stats = df.groupby('product_id').agg(
    review_count=('rating', 'count'),
    avg_rating=('rating', 'mean'),
    rating_std=('rating', 'std')
).reset_index()

# Filter to products with meaningful review counts
reliable_products = product_stats[product_stats['review_count'] >= 100]

print(f'📊 Products with 100+ reviews: {len(reliable_products):,}')
print(f'⭐ Average rating across all: {reliable_products["avg_rating"].mean():.2f}')
print(f'🔝 Highest rated: {reliable_products["avg_rating"].max():.2f}')
print(f'📉 Lowest rated: {reliable_products["avg_rating"].min():.2f}')

# Distribution of average ratings
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#1e1e2e')

ax.hist(reliable_products['avg_rating'], bins=30, 
        color=PINK, edgecolor='#2e2e42', alpha=0.8)
ax.axvline(reliable_products['avg_rating'].mean(), 
           color=GOLD, linestyle='--', linewidth=2,
           label=f'Mean: {reliable_products["avg_rating"].mean():.2f}')

ax.set_title('👑 Distribution of Average Product Ratings (100+ reviews)', 
             color='white', fontsize=14, pad=15)
ax.set_xlabel('Average Rating', color='white')
ax.set_ylabel('Number of Products', color='white')
ax.tick_params(colors='white')
ax.legend(facecolor='#1e1e2e', labelcolor='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2e2e42')

plt.tight_layout()
plt.savefig(f'{BASE}/visualizations/avg_rating_dist.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

### 9.5 — Demand Spikes Detection (Key Insight for Drop Queen!)

In [ ]:
# Identify demand spikes — months where reviews jumped significantly
monthly_reviews['pct_change'] = monthly_reviews['review_count'].pct_change() * 100
monthly_reviews['is_spike'] = monthly_reviews['pct_change'] > 20  # 20%+ increase = spike

spikes = monthly_reviews[monthly_reviews['is_spike']]
print(f'🔥 Demand spikes detected: {len(spikes)}')
print(f'📊 Average spike size: {spikes["pct_change"].mean():.1f}%')
print(f'🚀 Largest spike: {spikes["pct_change"].max():.1f}%')

# Plot with spikes highlighted
fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#1e1e2e')

ax.plot(range(len(monthly_reviews)), 
        monthly_reviews['review_count'],
        color=PURPLE, linewidth=1.5, alpha=0.7)

# Highlight spike months
spike_indices = monthly_reviews[monthly_reviews['is_spike']].index
ax.scatter(spike_indices, 
           monthly_reviews.loc[spike_indices, 'review_count'],
           color=PINK, s=80, zorder=5, label=f'Demand Spikes ({len(spikes)})')

ax.set_title('👑 Drop Queen — Demand Spike Detection', 
             color='white', fontsize=14, pad=15)
ax.set_xlabel('Month', color='white')
ax.set_ylabel('Review Volume', color='white')
ax.tick_params(colors='white')
ax.legend(facecolor='#1e1e2e', labelcolor='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#2e2e42')

plt.tight_layout()
plt.savefig(f'{BASE}/visualizations/demand_spikes.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print('✅ Demand spike analysis complete — this is what Drop Queen will PREDICT!')

## 💾 Step 10 — Save Cleaned Data to Google Drive

In [ ]:
PROCESSED = f'{BASE}/data/processed'

# Save main cleaned dataset
df.to_csv(f'{PROCESSED}/amazon_beauty_clean.csv', index=False)
print(f'✅ Cleaned dataset saved: {len(df):,} rows')

# Save product stats
product_stats.to_csv(f'{PROCESSED}/product_stats.csv', index=False)
print(f'✅ Product stats saved: {len(product_stats):,} products')

# Save monthly demand data
monthly_reviews.to_csv(f'{PROCESSED}/monthly_demand.csv', index=False)
print(f'✅ Monthly demand data saved: {len(monthly_reviews)} months')

# Save reliable products (100+ reviews)
reliable_products.to_csv(f'{PROCESSED}/reliable_products.csv', index=False)
print(f'✅ Reliable products saved: {len(reliable_products):,} products')

print('\n👑 All data saved to Google Drive!')

## 📊 Step 11 — EDA Summary Report

In [ ]:
print('=' * 60)
print('👑 DROP QUEEN — WEEK 1 EDA SUMMARY REPORT')
print('=' * 60)
print(f'\n📊 DATASET OVERVIEW')
print(f'   Total reviews analyzed:     {len(df):>12,}')
print(f'   Unique products tracked:    {df["product_id"].nunique():>12,}')
print(f'   Unique users:               {df["user_id"].nunique():>12,}')
print(f'   Date range:                 {df["date"].min().date()} to {df["date"].max().date()}')
print(f'\n⭐ RATINGS INSIGHTS')
print(f'   Average rating:             {df["rating"].mean():>12.2f}')
print(f'   5-star reviews:             {(df["rating"]==5).sum():>12,} ({(df["rating"]==5).mean()*100:.1f}%)')
print(f'   1-star reviews:             {(df["rating"]==1).sum():>12,} ({(df["rating"]==1).mean()*100:.1f}%)')
print(f'\n📈 DEMAND INSIGHTS')
print(f'   Total demand spikes:        {len(spikes):>12,}')
print(f'   Avg spike magnitude:        {spikes["pct_change"].mean():>11.1f}%')
print(f'   Max spike:                  {spikes["pct_change"].max():>11.1f}%')
print(f'   Peak demand month:          {monthly_reviews.loc[peak_idx, "year_month_str"]:>12}')
print(f'\n🏆 TOP PRODUCT')
print(f'   Most reviewed product:      {top_products.loc[0, "product_id"]}')
print(f'   Review count:               {top_products.loc[0, "review_count"]:>12,}')
print(f'\n✅ DATA QUALITY')
print(f'   Missing values:             {df.isnull().sum().sum():>12,}')
print(f'   Reliable products (100+):   {len(reliable_products):>12,}')
print('\n' + '=' * 60)
print('🚀 NEXT STEPS — Week 2: Feature Engineering')
print('   → Create lag features for time series forecasting')
print('   → Build rolling average demand features')
print('   → Engineer TikTok trend signal features')
print('   → Prepare data for Prophet & XGBoost models')
print('=' * 60)

## 🎉 Week 1 Complete!

Great work Chastity! You've completed Week 1 of Drop Queen. Here's what you accomplished:

| Task | Status |
|---|---|
| Google Drive connected | ✅ Done |
| Project folders created | ✅ Done |
| Libraries installed | ✅ Done |
| Kaggle dataset downloaded | ✅ Done |
| Data cleaned & preprocessed | ✅ Done |
| EDA visualizations created | ✅ Done |
| Demand spikes identified | ✅ Done |
| Clean data saved to Drive | ✅ Done |

---

### 🚀 Next Week — Week 2: Feature Engineering
Open notebook: `02_FeatureEngineering_DropQueen.ipynb`

---

> 👑 *Drop Queen — Built by Chastity Lewis | @LavishCreativeCo | Mercy University Spring 2026*